# BaTiO₃: training, polarization, BECs, and finite-field MD

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/01_batio3_workflow.ipynb)

This standalone workflow trains a compact MACEField model on a deterministic subset of the QE-labeled BaTiO₃ response data, then demonstrates Ti-displacement response and a short alternating-field molecular-dynamics cycle. The 135-atom configurations came from Flare active learning, not AIMD. The bundled QE labels are coherent zero-field energy, force, stress, polarization, BEC, and polarizability labels on the same structures.

The training subset is a short teaching example. The single-point and MD sections use the bundled specialist checkpoint, so their results do not depend on running or saving the training cell first. The available zero-field response data and short field cycle do not establish hysteresis or switching accuracy.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def ensure_checkout(path, url):
    path = Path(path)
    if (path / ".git").is_dir():
        branch = subprocess.run(
            ["git", "-C", str(path), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch == "main":
            return
        subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
        subprocess.run(
            ["git", "-C", str(path), "checkout", "-B", "main", "origin/main"],
            check=True,
        )
        return
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", url, str(path)],
        check=True,
    )


if IN_COLAB:
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    PACKAGE_REPO = Path("/content/mace-field")
    ensure_checkout(TUTORIAL_REPO, TUTORIAL_URL)
    ensure_checkout(PACKAGE_REPO, PACKAGE_URL)
else:
    CWD = Path.cwd().resolve()
    TUTORIAL_REPO = next(
        (
            path
            for path in (CWD, *CWD.parents)
            if (path / "tutorials/macefield/data").is_dir()
            and (path / "README.md").is_file()
        ),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run this notebook from a clone of mdi-group/mace-field-tutorial, "
            "or open it in Colab."
        )

    requested_repo = os.environ.get("MACEFIELD_REPO")
    if requested_repo:
        PACKAGE_REPO = Path(requested_repo).expanduser().resolve()
        if not (PACKAGE_REPO / "mace/modules/extensions.py").is_file():
            raise RuntimeError("MACEFIELD_REPO does not point to a MACE-Field checkout.")
        branch = subprocess.run(
            ["git", "-C", str(PACKAGE_REPO), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch != "main":
            raise RuntimeError("Set MACEFIELD_REPO to a checkout on the main branch.")
    else:
        candidates = [
            TUTORIAL_REPO.parent / "mace-field-main",
            TUTORIAL_REPO.parent / "mace-field",
        ]
        PACKAGE_REPO = None
        for candidate in candidates:
            if not (candidate / "mace/modules/extensions.py").is_file():
                continue
            branch = subprocess.run(
                ["git", "-C", str(candidate), "branch", "--show-current"],
                capture_output=True,
                text=True,
                check=False,
            ).stdout.strip()
            if branch == "main":
                PACKAGE_REPO = candidate.resolve()
                break
        if PACKAGE_REPO is None:
            PACKAGE_REPO = Path("/tmp/mace-field-main")
            ensure_checkout(PACKAGE_REPO, PACKAGE_URL)

os.environ["MACEFIELD_REPO"] = str(PACKAGE_REPO)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, str(PACKAGE_REPO))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--editable", str(PACKAGE_REPO)],
    check=True,
)
os.chdir(TUTORIAL_REPO)

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA_DIR = TUTORIAL_DIR / "data"
print("Tutorial repository:", TUTORIAL_REPO)
print("MACE-Field source:", PACKAGE_REPO)


## 1. Inspect the training subset

The 12/4 subset is sampled across the fixed training and validation split. This section checks the cell sizes, available labels, and zero-field tags before fitting.


In [ ]:
import numpy as np
from ase.io import read

TRAIN = DATA_DIR / "batio3/batio3_qe_train_sample.extxyz"
VALID = DATA_DIR / "batio3/batio3_qe_valid_sample.extxyz"
train = read(TRAIN, index=":")
valid = read(VALID, index=":")
print("train / validation frames:", len(train), "/", len(valid))
print("atoms per frame:", sorted({len(a) for a in train + valid}))
print("info label keys:", sorted(train[0].info))
print("array label keys:", sorted(k for k in train[0].arrays if k.startswith("REF_")))
print("electric fields:", np.unique(
    np.round([a.info["REF_electric_field"] for a in train + valid], 8), axis=0
))


## Train a compact response model

The default run is intentionally short for Colab. Increase `MACEFIELD_TUTORIAL_EPOCHS` for a longer demonstration. The small subset is for data and configuration practice, not for establishing converged accuracy.


In [ ]:
import os
import time
import yaml
import torch
import subprocess
import sys

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEFAULT_EPOCHS = 5 if DEVICE == "cuda" else 1
EPOCHS = int(os.environ.get("MACEFIELD_TUTORIAL_EPOCHS", DEFAULT_EPOCHS))
RUN_DIR = TUTORIAL_DIR / "runs" / f"batio3_qe_sample_{time.strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

config = {
    "name": "MACEField-BaTiO3-QE-sample",
    "model": "MACEField",
    "loss": "universal_field",
    "train_file": str(TRAIN),
    "valid_file": str(VALID),
    "valid_fraction": 0,
    "E0s": "average",
    "energy_key": "REF_energy",
    "forces_key": "REF_forces",
    "stress_key": "REF_stress",
    "electric_field_key": "REF_electric_field",
    "polarization_key": "REF_polarization",
    "becs_key": "REF_becs",
    "polarizability_key": "REF_polarizability",
    "compute_forces": True,
    "compute_stress": True,
    "compute_polarization": True,
    "compute_becs": True,
    "compute_polarizability": True,
    "energy_weight": 1.0,
    "forces_weight": 50.0,
    "stress_weight": 1.0,
    "polarization_weight": 10.0,
    "becs_weight": 50.0,
    "polarizability_weight": 10.0,
    "hidden_irreps": "16x0e + 16x1o",
    "num_interactions": 2,
    "max_L": 1,
    "r_max": 5.0,
    "batch_size": 1,
    "valid_batch_size": 1,
    "num_workers": 0,
    "lr": 0.005,
    "max_num_epochs": EPOCHS,
    "patience": max(3, EPOCHS),
    "eval_interval": 1,
    "seed": 23,
    "default_dtype": "float32",
    "device": DEVICE,
    "model_dir": str(RUN_DIR / "models"),
    "checkpoints_dir": str(RUN_DIR / "checkpoints"),
    "log_dir": str(RUN_DIR / "logs"),
    "results_dir": str(RUN_DIR / "results"),
    "work_dir": str(RUN_DIR / "work"),
}
for output_key in ("model_dir", "checkpoints_dir", "log_dir", "results_dir", "work_dir"):
    Path(config[output_key]).mkdir(parents=True, exist_ok=True)
CONFIG = RUN_DIR / "train.yml"
CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print("device:", DEVICE, "| epochs:", EPOCHS)
print("configuration:", CONFIG)
print("output model directory:", config["model_dir"])

subprocess.run(
    [sys.executable, "-m", "mace.cli.run_train", "--config", str(CONFIG)],
    cwd=PACKAGE_REPO,
    check=True,
)


## What this small fit can show

The model receives zero-field energy, force, stress, polarization, BEC, and polarizability supervision. The subsample is deliberately small and has no field reversal or switching-path labels. Use it to learn the MACEField data keys and response losses, not to report quantitative BaTiO₃ or hysteresis predictions.

## 2. Shift Ti and inspect polarization and Born effective charges

This section evaluates configurations around a selected Ti displacement and compares the predicted BEC with a finite-difference derivative of polarization. It loads the bundled BTO checkpoint directly.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read
from mace.calculators import MACECalculator

MODEL = DATA_DIR / "batio3/MACE-Field-BaTiO3.model"
SAMPLE = DATA_DIR / "macefield_batio3_reference.extxyz"
source = read(SAMPLE, index=0)
volume = source.get_volume()

symbols = np.asarray(source.get_chemical_symbols())
ti_candidates = np.flatnonzero(symbols == "Ti")
fractional = source.get_scaled_positions(wrap=True)
center_delta = fractional[ti_candidates] - 0.5
center_delta -= np.rint(center_delta)
ti_index = int(ti_candidates[np.argmin(np.linalg.norm(center_delta, axis=1))])
source.info.clear()
for key in list(source.arrays):
    if key not in {"numbers", "positions"}:
        source.arrays.pop(key)


def structure_at(displacement):
    atoms = source.copy()
    atoms.positions[ti_index, 2] += float(displacement)
    atoms.info.clear()
    return atoms


device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
calc = MACECalculator(
    model_paths=str(MODEL),
    model_type="MACEField",
    device=device,
    default_dtype="float32",
    electric_field=[0.0, 0.0, 0.0],
    compute_polarization=True,
    compute_becs=True,
    compute_polarizability=False,
)

displacements = np.linspace(-0.12, 0.12, 7)
fields = [-0.0036, 0.0, 0.0036]
polarizations = {field: [] for field in fields}
ti_zz = []

for displacement in displacements:
    atoms = structure_at(displacement)
    calc.electric_field = [0.0, 0.0, 0.0]
    calc.reset()
    atoms.calc = calc
    atoms.get_potential_energy()
    becs = np.asarray(calc.results["becs"]).reshape(len(atoms), 3, 3)
    ti_zz.append(float(becs[ti_index, 2, 2]))

    for field_z in fields:
        calc.electric_field = [0.0, 0.0, float(field_z)]
        calc.reset()
        atoms.calc = calc
        atoms.get_potential_energy()
        polarizations[field_z].append(float(calc.results["polarization"][2]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for field_z in fields:
    axes[0].plot(displacements, polarizations[field_z], "o-", label=f"Ez={field_z:+.4f} V/Å")
axes[0].set(xlabel="selected Ti displacement along z (Å)", ylabel="Pz (e/Å²)",
            title="Field-dependent polarization")
axes[0].legend()
axes[1].plot(displacements, ti_zz, "o-", color="#9b4f96")
axes[1].set(xlabel="selected Ti displacement along z (Å)", ylabel="Ti Z*zz (e)",
            title="Predicted Born effective charge")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## Check the BEC against a finite difference

For a cell polarization density, the Born charge includes the cell volume: $Z^*_{zz}=\Omega\,\partial P_z/\partial r_{\mathrm{Ti},z}$. This checks the model derivative against a central finite difference at zero field.


In [ ]:
delta = 1.0e-3
d0 = 0.04
field_z = 0.0

atoms0 = structure_at(d0)
calc.electric_field = [0.0, 0.0, field_z]
calc.reset()
atoms0.calc = calc
atoms0.get_potential_energy()
z_model = float(
    np.asarray(calc.results["becs"]).reshape(len(atoms0), 3, 3)[ti_index, 2, 2]
)

p_values = []
for displacement in (d0 - delta, d0 + delta):
    atoms = structure_at(displacement)
    calc.electric_field = [0.0, 0.0, field_z]
    calc.reset()
    atoms.calc = calc
    atoms.get_potential_energy()
    p_values.append(float(calc.results["polarization"][2]))

z_fd = volume * (p_values[1] - p_values[0]) / (2 * delta)
print("model Ti Z*zz (e):", z_model)
print("finite-difference Ti Z*zz (e):", z_fd)


## 3. Alternate a small electric field during MD

The short Langevin trajectory cycles the field up and down at two temperatures. It is an executable response demonstration, not a converged hysteresis or switching calculation.


## Set up a two-temperature field cycle

The small field interval is centered on zero and uses the finite-field response scale of the reference calculations. The trajectory is intentionally short for Colab. It can show how to change fields during MD, but the data do not validate hysteresis or polarization switching.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase import units
from ase.io import read
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from mace.calculators import MACECalculator

MODEL = DATA_DIR / "batio3/MACE-Field-BaTiO3.model"
SAMPLE = DATA_DIR / "macefield_batio3_reference.extxyz"
source = read(SAMPLE, index=0)
source.info.clear()
for key in list(source.arrays):
    if key not in {"numbers", "positions"}:
        source.arrays.pop(key)

field_up = np.linspace(-0.0036, 0.0036, 5)
field_cycle = np.concatenate([field_up, field_up[-2::-1]])
temperatures_K = [25.0, 200.0]
steps_per_field = 5
dt_fs = 0.5
friction_per_fs = 0.02
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"

records_by_temperature = {}
for temperature_K in temperatures_K:
    atoms = source.copy()
    rng = np.random.default_rng(23 + int(temperature_K))
    MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K, rng=rng)
    Stationary(atoms)

    calc = MACECalculator(
        model_paths=str(MODEL),
        model_type="MACEField",
        device=device,
        default_dtype="float32",
        electric_field=[0.0, 0.0, float(field_cycle[0])],
        compute_polarization=True,
        compute_becs=False,
        compute_polarizability=False,
    )
    atoms.calc = calc
    dyn = Langevin(
        atoms,
        timestep=dt_fs * units.fs,
        temperature_K=temperature_K,
        friction=friction_per_fs / units.fs,
        rng=rng,
    )

    polarization_z = []
    energy = []
    for field_z in field_cycle:
        calc.electric_field = [0.0, 0.0, float(field_z)]
        calc.reset()
        dyn.run(steps_per_field)
        energy.append(atoms.get_potential_energy())
        polarization_z.append(float(calc.results["polarization"][2]))

    records_by_temperature[temperature_K] = {
        "field_z": field_cycle.copy(),
        "polarization_z": np.asarray(polarization_z),
        "energy": np.asarray(energy),
    }
    print(
        f"{temperature_K:g} K: {len(field_cycle) * steps_per_field} MD steps, "
        f"final energy {energy[-1]:.6f} eV"
    )


## Plot polarization during the field cycle

Polarization is projected on z and converted to microcoulombs per square centimetre. A separation between the forward and reverse traces is a finite-rate trajectory effect here; it should not be interpreted as a validated ferroelectric hysteresis loop.


In [ ]:
P_TO_UC_CM2 = 1602.176634
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
n_up = len(field_up)
for temperature_K, data in records_by_temperature.items():
    p = data["polarization_z"] * P_TO_UC_CM2
    axes[0].plot(data["field_z"][:n_up], p[:n_up], "o-", ms=3,
                 label=f"{temperature_K:g} K, increasing")
    axes[0].plot(data["field_z"][n_up - 1:], p[n_up - 1:], "o--", ms=3,
                 label=f"{temperature_K:g} K, decreasing")
    axes[1].plot(np.arange(len(p)) * steps_per_field * dt_fs, p,
                 label=f"{temperature_K:g} K")
axes[0].set(xlabel="Ez (V/Å)", ylabel="Pz (µC/cm²)",
            title="Small-field response cycle")
axes[1].set(xlabel="time (fs)", ylabel="Pz (µC/cm²)",
            title="Polarization during the cycle")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()


## Interpretation

This is a short workflow demonstration using a real QE-trained potential. It samples only a few field-cycle steps and starts from one 135-atom structure. It cannot establish equilibrium dielectric response, coercive fields, domain nucleation, or hysteresis. Those claims require DFT reference configurations and response labels across both field polarities and the switching path, plus larger cells and convergence checks.